# Gold Skill Demand by Sector Mart

**Purpose**: Skill demand analysis across all sectors.

**Target Table**: `workspace.gold.gold_skill_demand_by_sector`

**Grain**: One row per sector per skill per date

**Sector Support**: Multi-sector enabled via sector_sk dimension

**Key Metrics**:
- Job count requiring skill
- Demand rank within sector
- Growth rate (WoW, MoM)
- Skill penetration rate
- Emerging vs declining skills

**Data Sources**:
- `workspace.warehouse.bridge_job_skill`
- `workspace.warehouse.fact_job_postings`
- `workspace.warehouse.dim_skill`
- `workspace.warehouse.dim_sector`

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_skill_demand_by_sector
USING DELTA
COMMENT 'Skill demand analysis by sector'
AS

WITH base_skill_demand AS (
  SELECT
    f.posting_date_sk AS trend_date_sk,
    f.sector_sk,
    bjs.skill_sk,
    bjs.skill_importance,
    
    -- MEASURES
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.job_sk END) AS job_count,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.company_sk END) AS company_count,
    
    -- Importance breakdown
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE AND bjs.skill_importance = 'REQUIRED' THEN f.job_sk END) AS required_count,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE AND bjs.skill_importance = 'PREFERRED' THEN f.job_sk END) AS preferred_count,
    
    AVG(bjs.extraction_confidence) AS avg_confidence
    
  FROM workspace.warehouse.fact_job_postings f
  INNER JOIN workspace.warehouse.bridge_job_skill bjs ON f.job_sk = bjs.job_sk
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
    AND f.sector_sk IS NOT NULL  -- Ensure valid sector assignment
    AND bjs.skill_sk IS NOT NULL
  GROUP BY f.posting_date_sk, f.sector_sk, bjs.skill_sk, bjs.skill_importance
),

aggregated_by_sector_skill_date AS (
  SELECT
    bsd.trend_date_sk,
    bsd.sector_sk,
    bsd.skill_sk,
    
    -- Aggregate measures
    SUM(bsd.job_count) AS total_job_count,
    SUM(bsd.company_count) AS total_company_count,
    SUM(bsd.required_count) AS total_required,
    SUM(bsd.preferred_count) AS total_preferred,
    AVG(bsd.avg_confidence) AS avg_extraction_confidence
    
  FROM base_skill_demand bsd
  GROUP BY bsd.trend_date_sk, bsd.sector_sk, bsd.skill_sk
),

with_rankings AS (
  SELECT
    agg.*,
    
    -- Rank skills within each sector and date
    DENSE_RANK() OVER (
      PARTITION BY agg.trend_date_sk, agg.sector_sk
      ORDER BY agg.total_job_count DESC
    ) AS demand_rank,
    
    -- Prior period values for growth calculation
    LAG(agg.total_job_count, 7) OVER (
      PARTITION BY agg.sector_sk, agg.skill_sk
      ORDER BY agg.trend_date_sk
    ) AS prev_week_job_count,
    
    LAG(agg.total_job_count, 30) OVER (
      PARTITION BY agg.sector_sk, agg.skill_sk
      ORDER BY agg.trend_date_sk
    ) AS prev_month_job_count,
    
    -- Rolling averages
    AVG(agg.total_job_count) OVER (
      PARTITION BY agg.sector_sk, agg.skill_sk
      ORDER BY agg.trend_date_sk
      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg,
    
    AVG(agg.total_job_count) OVER (
      PARTITION BY agg.sector_sk, agg.skill_sk
      ORDER BY agg.trend_date_sk
      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) AS rolling_30day_avg
    
  FROM aggregated_by_sector_skill_date agg
),

sector_totals AS (
  SELECT
    trend_date_sk,
    sector_sk,
    SUM(total_job_count) AS sector_total_jobs
  FROM with_rankings
  GROUP BY trend_date_sk, sector_sk
)

SELECT
  -- DIMENSIONS (sector_sk first for partitioning)
  wr.sector_sk,
  wr.skill_sk,
  wr.trend_date_sk,
  
  -- MEASURES
  wr.total_job_count AS job_count,
  wr.total_company_count AS company_count,
  wr.total_required AS required_count,
  wr.total_preferred AS preferred_count,
  wr.demand_rank,
  
  -- TEMPORAL METRICS
  CAST(wr.rolling_7day_avg AS DECIMAL(10,2)) AS rolling_7day_avg,
  CAST(wr.rolling_30day_avg AS DECIMAL(10,2)) AS rolling_30day_avg,
  
  -- DERIVED: Growth rates
  CASE 
    WHEN wr.prev_week_job_count > 0 
    THEN CAST((wr.total_job_count - wr.prev_week_job_count) AS DECIMAL(10,2)) / wr.prev_week_job_count
    ELSE NULL 
  END AS growth_rate_wow,
  
  CASE 
    WHEN wr.prev_month_job_count > 0 
    THEN CAST((wr.total_job_count - wr.prev_month_job_count) AS DECIMAL(10,2)) / wr.prev_month_job_count
    ELSE NULL 
  END AS growth_rate,
  
  -- DERIVED: Penetration rate (skill usage as % of sector jobs)
  CASE 
    WHEN st.sector_total_jobs > 0 
    THEN CAST(wr.total_job_count AS DECIMAL(10,4)) / st.sector_total_jobs
    ELSE NULL 
  END AS penetration_rate,
  
  -- DERIVED: Required percentage
  CASE 
    WHEN wr.total_job_count > 0 
    THEN CAST(wr.total_required AS DECIMAL(10,4)) / wr.total_job_count
    ELSE NULL 
  END AS required_pct,
  
  -- Quality metric
  CAST(wr.avg_extraction_confidence AS DECIMAL(5,4)) AS avg_confidence,
  
  -- METADATA
  CURRENT_TIMESTAMP() AS updated_at,
  UUID() AS batch_id
  
FROM with_rankings wr
INNER JOIN sector_totals st ON wr.trend_date_sk = st.trend_date_sk AND wr.sector_sk = st.sector_sk
WHERE wr.demand_rank <= 500  -- Keep top 500 skills per sector per day
ORDER BY wr.sector_sk, wr.trend_date_sk DESC, wr.demand_rank;


In [0]:
%sql
-- Validation: Top skills by sector
SELECT
  s.sector_name,
  sk.skill_name,
  sk.skill_category,
  SUM(gsds.job_count) AS total_jobs,
  ROUND(AVG(gsds.penetration_rate), 4) AS avg_penetration_rate,
  ROUND(AVG(gsds.growth_rate), 4) AS avg_growth_rate,
  ROUND(AVG(gsds.required_pct), 4) AS avg_required_pct
FROM workspace.gold.gold_skill_demand_by_sector gsds
INNER JOIN workspace.warehouse.dim_sector s ON gsds.sector_sk = s.sector_sk
INNER JOIN workspace.warehouse.dim_skill sk ON gsds.skill_sk = sk.skill_sk
WHERE gsds.trend_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 30), 'yyyyMMdd') AS INT)
  AND gsds.demand_rank <= 20  -- Top 20 skills per sector
GROUP BY s.sector_name, sk.skill_name, sk.skill_category
ORDER BY s.sector_name, total_jobs DESC
LIMIT 100;
